In [ ]:
import pandas as pd
import datetime
import yaml
import pyotp
import ta
from NorenRestApiPy.NorenApi import NorenApi

class ShoonyaApiPy(NorenApi):
    def __init__(self):
        super().__init__(host='https://api.shoonya.com/NorenWClientTP/', websocket='wss://api.shoonya.com/NorenWSTP/')

# Load credentials
with open('cred.yml') as f:
    cred = yaml.load(f, Loader=yaml.FullLoader)

# Generate OTP for login
TOKEN = cred['factor2']
otp = pyotp.TOTP(TOKEN).now()

# Initialize API and Login
api = ShoonyaApiPy()
ret = api.login(
    userid=cred['user'],
    password=cred['pwd'],
    twoFA=otp,
    vendor_code=cred['vc'],
    api_secret=cred['apikey'],
    imei=cred['imei']
)

if not ret:
    print("❌ Login Failed")
    exit()

print("✅ Login Successful")

# Get today's date and reset time to 00:00:00
today = datetime.datetime.today().replace(hour=0, minute=0, second=0, microsecond=0)

# Calculate the last 5 business days (excluding weekends)
days_back = 365
lastBusDay = today
while days_back > 0:
    lastBusDay -= datetime.timedelta(days=1)
    if lastBusDay.weekday() < 5:  # Monday to Friday are business days
        days_back -= 1

# Convert last business day to timestamp
start_timestamp = int(lastBusDay.timestamp())

# List of stocks to fetch data for
# stocks_arr = ['SAMMAANCAP-EQ', 'RELIANCE-EQ', 'TCS-EQ', 'INFY-EQ']  # Add your stocks here
stocks_arr =['SAMMAANCAP-EQ','360ONE-EQ', '3MINDIA-EQ', 'ABB-EQ', 'ACC-EQ', 'AIAENG-EQ', 'APLAPOLLO-EQ', 'AUBANK-EQ', 'AARTIIND-EQ', 'AAVAS-EQ', 'ABBOTINDIA-EQ', 'ACE-EQ', 'ADANIENSOL-EQ', 'ADANIENT-EQ', 
         'ADANIGREEN-EQ', 'ADANIPORTS-EQ', 'ADANIPOWER-EQ', 'ATGL-EQ', 'AWL-EQ', 'ABCAPITAL-EQ', 'ABFRL-EQ', 'AEGISCHEM-EQ', 'AETHER-EQ', 'AFFLE-EQ', 'AJANTPHARM-EQ', 'APLLTD-EQ', 'ALKEM-EQ', 
         'ALKYLAMINE-EQ', 'ALLCARGO-EQ', 'ALOKINDS-EQ', 'ARE&M-EQ', 'AMBER-EQ', 'AMBUJACEM-EQ', 'ANANDRATHI-EQ', 'ANGELONE-EQ', 'ANURAS-EQ', 'APARINDS-EQ', 'APOLLOHOSP-EQ', 'APOLLOTYRE-EQ', 'APTUS-EQ', 'ACI-EQ', 'ASAHIINDIA-EQ', 'ASHOKLEY-EQ', 
         'ASIANPAINT-EQ', 'ASTERDM-EQ', 'ASTRAZEN-EQ', 'ASTRAL-EQ', 'ATUL-EQ', 'AUROPHARMA-EQ', 'AVANTIFEED-EQ', 'DMART-EQ', 'AXISBANK-EQ', 'BEML-EQ', 'BLS-EQ', 'BSE-EQ', 'BAJAJ-AUTO-EQ', 'BAJFINANCE-EQ', 'BAJAJFINSV-EQ', 'BAJAJHLDNG-EQ', 'BALAMINES-EQ', 'BALKRISIND-EQ', 
         'BALRAMCHIN-EQ', 'BANDHANBNK-EQ', 'BANKBARODA-EQ', 'BANKINDIA-EQ', 'MAHABANK-EQ', 'BATAINDIA-EQ', 'BAYERCROP-EQ', 'BERGEPAINT-EQ', 'BDL-EQ', 'BEL-EQ', 'BHARATFORG-EQ', 'BHEL-EQ', 'BPCL-EQ', 'BHARTIARTL-EQ', 'BIKAJI-EQ', 'BIOCON-EQ', 'BIRLACORPN-EQ', 'BSOFT-EQ', 'BLUEDART-EQ', 'BLUESTARCO-EQ', 
         'BBTC-EQ', 'BORORENEW-EQ', 'BOSCHLTD-EQ', 'BRIGADE-EQ', 'BRITANNIA-EQ', 'MAPMYINDIA-EQ', 'CCL-EQ', 'CESC-EQ', 'CGPOWER-EQ', 'CIEINDIA-EQ', 'CRISIL-EQ', 'CSBBANK-EQ', 'CAMPUS-EQ', 'CANFINHOME-EQ', 'CANBK-EQ', 'CAPLIPOINT-EQ', 'CGCL-EQ', 'CARBORUNIV-EQ', 'CASTROLIND-EQ', 'CEATLTD-EQ', 'CELLO-EQ',
         'CENTRALBK-EQ', 'CDSL-EQ', 'CENTURYPLY-EQ', 'CENTURYTEX-EQ', 'CERA-EQ', 'CHALET-EQ', 'CHAMBLFERT-EQ', 'CHEMPLASTS-EQ', 'CHENNPETRO-EQ', 'CHOLAHLDNG-EQ', 'CHOLAFIN-EQ', 'CIPLA-EQ', 'CUB-EQ', 'CLEAN-EQ', 'COALINDIA-EQ', 'COCHINSHIP-EQ', 'COFORGE-EQ', 'COLPAL-EQ', 'CAMS-EQ', 'CONCORDBIO-EQ', 'CONCOR-EQ', 
         'COROMANDEL-EQ', 'CRAFTSMAN-EQ', 'CREDITACC-EQ', 'CROMPTON-EQ', 'CUMMINSIND-EQ', 'CYIENT-EQ', 'DCMSHRIRAM-EQ', 'DLF-EQ', 'DOMS-EQ', 'DABUR-EQ', 'DALBHARAT-EQ', 'DATAPATTNS-EQ', 'DEEPAKFERT-EQ', 'DEEPAKNTR-EQ', 'DELHIVERY-EQ', 'DEVYANI-EQ', 'DIVISLAB-EQ', 'DIXON-EQ', 'LALPATHLAB-EQ', 'DRREDDY-EQ', 'EIDPARRY-EQ',
         'EIHOTEL-EQ', 'EPL-EQ', 'EASEMYTRIP-EQ', 'EICHERMOT-EQ', 'ELECON-EQ', 'ELGIEQUIP-EQ', 'EMAMILTD-EQ', 'ENDURANCE-EQ', 'ENGINERSIN-EQ', 'EQUITASBNK-EQ', 'ERIS-EQ', 'ESCORTS-EQ', 'EXIDEIND-EQ', 'FDC-EQ', 'NYKAA-EQ', 'FEDERALBNK-EQ', 'FACT-EQ', 'FINEORG-EQ', 'FINCABLES-EQ', 'FINPIPE-EQ', 'FSL-EQ', 'FIVESTAR-EQ',
         'FORTIS-EQ', 'GAIL-EQ', 'GMMPFAUDLR-EQ', 'GMRINFRA-EQ', 'GRSE-EQ', 'GICRE-EQ', 'GILLETTE-EQ', 'GLAND-EQ', 'GLAXO-EQ', 'GLS-EQ', 'GLENMARK-EQ', 'MEDANTA-EQ', 'GPIL-EQ', 'GODFRYPHLP-EQ', 'GODREJCP-EQ', 'GODREJIND-EQ', 'GODREJPROP-EQ', 'GRANULES-EQ', 'GRAPHITE-EQ', 'GRASIM-EQ', 'GESHIP-EQ', 'GRINDWELL-EQ', 'GAEL-EQ',
         'FLUOROCHEM-EQ', 'GUJGASLTD-EQ', 'GMDCLTD-EQ', 'GNFC-EQ', 'GPPL-EQ', 'GSFC-EQ', 'GSPL-EQ', 'HEG-EQ', 'HBLPOWER-EQ', 'HCLTECH-EQ', 'HDFCAMC-EQ', 'HDFCBANK-EQ', 'HDFCLIFE-EQ', 'HFCL-EQ', 'HAPPSTMNDS-EQ', 'HAPPYFORGE-EQ', 'HAVELLS-EQ', 'HEROMOTOCO-EQ', 'HSCL-EQ', 'HINDALCO-EQ', 'HAL-EQ', 'HINDCOPPER-EQ', 'HINDPETRO-EQ',
         'HINDUNILVR-EQ', 'HINDZINC-EQ', 'POWERINDIA-EQ', 'HOMEFIRST-EQ', 'HONASA-EQ', 'HONAUT-EQ', 'HUDCO-EQ', 'ICICIBANK-EQ', 'ICICIGI-EQ', 'ICICIPRULI-EQ', 'ISEC-EQ', 'IDBI-EQ', 'IDFCFIRSTB-EQ', 'IDFC-EQ', 'IIFL-EQ', 'IRB-EQ', 'IRCON-EQ', 'ITC-EQ', 'ITI-EQ', 'INDIACEM-EQ', 'IBULHSGFIN-EQ', 'INDIAMART-EQ', 'INDIANB-EQ', 'IEX-EQ',
         'INDHOTEL-EQ', 'IOC-EQ', 'IOB-EQ', 'IRCTC-EQ', 'IRFC-EQ', 'INDIGOPNTS-EQ', 'IGL-EQ', 'INDUSTOWER-EQ', 'INDUSINDBK-EQ', 'NAUKRI-EQ', 'INFY-EQ', 'INOXWIND-EQ', 'INTELLECT-EQ', 'INDIGO-EQ', 'IPCALAB-EQ', 'JBCHEPHARM-EQ', 'JKCEMENT-EQ', 'JBMA-EQ', 'JKLAKSHMI-EQ', 'JKPAPER-EQ', 'JMFINANCIL-EQ', 'JSWENERGY-EQ', 'JSWINFRA-EQ', 'JSWSTEEL-EQ', 'JAIBALAJI-EQ', 'J&KBANK-EQ', 'JINDALSAW-EQ', 'JSL-EQ', 'JINDALSTEL-EQ', 'JIOFIN-EQ', 'JUBLFOOD-EQ', 'JUBLINGREA-EQ', 'JUBLPHARMA-EQ', 'JWL-EQ', 'JUSTDIAL-EQ',
         'JYOTHYLAB-EQ', 'KPRMILL-EQ', 'KEI-EQ', 'KNRCON-EQ', 'KPITTECH-EQ', 'KRBL-EQ', 'KSB-EQ', 'KAJARIACER-EQ', 'KPIL-EQ', 'KALYANKJIL-EQ', 'KANSAINER-EQ', 'KARURVYSYA-EQ', 'KAYNES-EQ', 'KEC-EQ', 'KFINTECH-EQ', 'KOTAKBANK-EQ', 'KIMS-EQ', 'L&TFH-EQ', 'LTTS-EQ', 'LICHSGFIN-EQ', 'LTIM-EQ', 'LT-EQ', 'LATENTVIEW-EQ', 'LAURUSLABS-EQ', 'LXCHEM-EQ', 'LEMONTREE-EQ', 'LICI-EQ', 'LINDEINDIA-EQ', 'LLOYDSME-EQ', 'LUPIN-EQ', 'MMTC-EQ', 'MRF-EQ', 'MTARTECH-EQ', 'LODHA-EQ', 'MGL-EQ', 'MAHSEAMLES-EQ', 'M&MFIN-EQ', 'M&M-EQ', 'MHRIL-EQ', 'MAHLIFE-EQ', 'MANAPPURAM-EQ', 'MRPL-EQ', 'MANKIND-EQ', 'MARICO-EQ', 'MARUTI-EQ', 'MASTEK-EQ', 'MFSL-EQ', 'MAXHEALTH-EQ', 'MAZDOCK-EQ', 'MEDPLUS-EQ', 'METROBRAND-EQ', 'METROPOLIS-EQ', 'MINDACORP-EQ', 'MSUMI-EQ', 'MOTILALOFS-EQ', 'MPHASIS-EQ', 'MCX-EQ', 'MUTHOOTFIN-EQ', 'NATCOPHARM-EQ', 'NBCC-EQ', 'NCC-EQ', 'NHPC-EQ', 'NLCINDIA-EQ', 'NMDC-EQ', 'NSLNISP-EQ', 'NTPC-EQ', 'NH-EQ', 'NATIONALUM-EQ', 'NAVINFLUOR-EQ', 'NESTLEIND-EQ', 'NETWORK18-EQ', 'NAM-INDIA-EQ', 'NUVAMA-EQ', 'NUVOCO-EQ', 'OBEROIRLTY-EQ', 'ONGC-EQ', 'OIL-EQ', 'OLECTRA-EQ', 'PAYTM-EQ', 'OFSS-EQ', 'POLICYBZR-EQ', 'PCBL-EQ', 'PIIND-EQ', 'PNBHOUSING-EQ', 'PNCINFRA-EQ', 'PVRINOX-EQ', 'PAGEIND-EQ',
         'PATANJALI-EQ', 'PERSISTENT-EQ', 'PETRONET-EQ', 'PHOENIXLTD-EQ', 'PIDILITIND-EQ', 'PEL-EQ', 'PPLPHARMA-EQ', 'POLYMED-EQ', 'POLYCAB-EQ', 'POONAWALLA-EQ', 'PFC-EQ', 'POWERGRID-EQ', 'PRAJIND-EQ', 'PRESTIGE-EQ', 'PRINCEPIPE-EQ', 'PRSMJOHNSN-EQ', 'PGHH-EQ', 'PNB-EQ', 'QUESS-EQ', 'RRKABEL-EQ', 'RBLBANK-EQ', 'RECLTD-EQ', 'RHIM-EQ', 'RITES-EQ', 'RADICO-EQ', 'RVNL-EQ', 'RAILTEL-EQ', 'RAINBOW-EQ', 'RAJESHEXPO-EQ', 'RKFORGE-EQ', 'RCF-EQ', 'RATNAMANI-EQ', 'RTNINDIA-EQ', 'RAYMOND-EQ', 'REDINGTON-EQ', 'RELIANCE-EQ', 'RBA-EQ', 'ROUTE-EQ', 'SBFC-EQ', 'SBICARD-EQ', 'SBILIFE-EQ', 'SJVN-EQ', 'SKFINDIA-EQ', 'SRF-EQ', 'SAFARI-EQ', 'MOTHERSON-EQ', 'SANOFI-EQ', 'SAPPHIRE-EQ', 'SAREGAMA-EQ', 'SCHAEFFLER-EQ', 'SCHNEIDER-EQ', 'SHREECEM-EQ', 'RENUKA-EQ', 'SHRIRAMFIN-EQ', 'SHYAMMETL-EQ', 'SIEMENS-EQ', 'SIGNATURE-EQ', 'SOBHA-EQ', 'SOLARINDS-EQ', 'SONACOMS-EQ', 'SONATSOFTW-EQ', 'STARHEALTH-EQ', 'SBIN-EQ', 'SAIL-EQ', 'SWSOLAR-EQ', 'STLTECH-EQ', 'SUMICHEM-EQ', 'SPARC-EQ', 'SUNPHARMA-EQ', 'SUNTV-EQ', 'SUNDARMFIN-EQ', 'SUNDRMFAST-EQ', 'SUNTECK-EQ', 'SUPREMEIND-EQ', 'SUVENPHAR-EQ', 'SUZLON-EQ',
         'SWANENERGY-EQ', 'SYNGENE-EQ', 'SYRMA-EQ', 'TV18BRDCST-EQ', 'TVSMOTOR-EQ', 'TVSSCS-EQ', 'TMB-EQ', 'TANLA-EQ', 'TATACHEM-EQ', 'TATACOMM-EQ', 'TCS-EQ', 'TATACONSUM-EQ', 'TATAELXSI-EQ', 'TATAINVEST-EQ', 'TATAMTRDVR-EQ', 'TATAMOTORS-EQ', 'TATAPOWER-EQ', 'TATASTEEL-EQ', 'TATATECH-EQ', 'TTML-EQ', 'TECHM-EQ', 'TEJASNET-EQ', 'NIACL-EQ', 'RAMCOCEM-EQ', 'THERMAX-EQ', 'TIMKEN-EQ', 'TITAGARH-EQ', 'TITAN-EQ', 'TORNTPHARM-EQ', 'TORNTPOWER-EQ', 'TRENT-EQ', 'TRIDENT-EQ', 'TRIVENI-EQ', 'TRITURBINE-EQ', 'TIINDIA-EQ', 'UCOBANK-EQ', 'UNOMINDA-EQ', 'UPL-EQ', 'UTIAMC-EQ', 'UJJIVANSFB-EQ', 'ULTRACEMCO-EQ', 'UNIONBANK-EQ', 'UBL-EQ', 'MCDOWELL-N-EQ', 'USHAMART-EQ', 'VGUARD-EQ', 'VIPIND-EQ', 'VAIBHAVGBL-EQ', 'VTL-EQ', 'VARROC-EQ', 'VBL-EQ', 'MANYAVAR-EQ', 'VEDL-EQ', 
         'VIJAYA-EQ', 'IDEA-EQ', 'VOLTAS-EQ', 'WELCORP-EQ', 'WELSPUNLIV-EQ', 'WESTLIFE-EQ', 'WHIRLPOOL-EQ', 'WIPRO-EQ', 'YESBANK-EQ', 'ZFCVINDIA-EQ', 'ZEEL-EQ', 'ZENSARTECH-EQ', 'ZOMATO-EQ', 'ZYDUSLIFE-EQ', 'ECLERX-EQ']


save_path = r'D:\DATA_Stocks'

for stock in stocks_arr:
    print(f"Fetching data for {stock}...")
    ret = api.get_time_price_series(
        exchange='NSE',
        token=stock,
        starttime=start_timestamp,
        interval=15
    )
    
    if ret:
        df = pd.DataFrame(ret)
        required_columns = ['time', 'into', 'inth', 'intl', 'intc', 'intvwap', 'intv']
        df = df[required_columns]
        df['time'] = pd.to_datetime(df['time'], dayfirst=True)
        
        # Rename columns for clarity
        df.rename(columns={'into': 'open', 'inth': 'high', 'intl': 'low', 'intc': 'close', 'intvwap': 'vwap', 'intv': 'volume'}, inplace=True)
        
        # Ensure numeric data types
        df[['open', 'high', 'low', 'close', 'vwap', 'volume']] = df[['open', 'high', 'low', 'close', 'vwap', 'volume']].apply(pd.to_numeric, errors='coerce')
        
        # Compute indicators
        df['upper_bb'] = ta.volatility.BollingerBands(df['close'], window=20, window_dev=2).bollinger_hband()
        df['lower_bb'] = ta.volatility.BollingerBands(df['close'], window=20, window_dev=2).bollinger_lband()
        df['rsi'] = ta.momentum.RSIIndicator(df['close'], window=14).rsi()
        df['adx'] = ta.trend.ADXIndicator(df['high'], df['low'], df['close'], window=14).adx()
        
        # Save updated data (overwrite original file)
        filename = f'{stock}.xlsx'
        full_path = f'{save_path}\\{filename}'
        df.to_excel(full_path, index=False)
        print(f"Updated data saved to: {full_path}")
    else:
        print(f"No data received for {stock}.")


In [ ]:
import pandas as pd
import ta

# Load historical data from Excel file
def load_data(file_path, sheet_name="Sheet1"):
    df = pd.read_excel(file_path, sheet_name=sheet_name)
    df["time"] = pd.to_datetime(df["time"])
    df = df.sort_values("time")
    return df

# Backtesting function
def run_backtest(df):
    signals = []
    for i in range(6, len(df)):
        if (
            df['low'].iloc[i-2] >= df['upper_bb'].iloc[i-2] and
            df['high'].iloc[i-2] >= df['upper_bb'].iloc[i-2] and
            df['rsi'].iloc[i-1] >= 60 and
            df['open'].iloc[i-6] / df['close'].iloc[i-1] >= 0.955 and
            df['close'].iloc[i-1] <= df['upper_bb'].iloc[i-1] and
            df['volume'].iloc[i-1] >= 30000 and
            70 <= df['close'].iloc[i-1] <= 5000 and
            df['close'].iloc[i-1] <= df['open'].iloc[i-1] and
            df['close'].iloc[i-1] <= df['vwap'].iloc[i-1] and
            df['adx'].iloc[i-1] >= 26
        ):
            signals.append(df[["time", "close"]].iloc[i])  # Store time and close price for reference
    return pd.DataFrame(signals)

# Main function
def main():
    file_path = r"D:\DATA_Stocks\AARTIIND-EQ.xlsx"  # Use raw string to avoid escape issues
    output_path = r"D:\DATA_Stocks\RESULTS\backtest_results.xlsx"
    
    df = load_data(file_path)
    signals_df = run_backtest(df)
    signals_df.to_excel(output_path, index=False)
    print(f"Backtest completed. Results saved in {output_path}")

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import os
import glob
import ta

# Load historical data from Excel file
def load_data(file_path, sheet_name="Sheet1"):
    df = pd.read_excel(file_path, sheet_name=sheet_name)
    df["time"] = pd.to_datetime(df["time"])
    df = df.sort_values("time")
    return df

# Backtesting function
def run_backtest(df, stock_name):
    signals = []
    for i in range(6, len(df)):
        if (
            df['low'].iloc[i-2] >= df['upper_bb'].iloc[i-2] and
            df['high'].iloc[i-2] >= df['upper_bb'].iloc[i-2] and
            df['rsi'].iloc[i-1] >= 60 and
            df['open'].iloc[i-6] / df['close'].iloc[i-1] >= 0.955 and
            df['close'].iloc[i-1] <= df['upper_bb'].iloc[i-1] and
            df['volume'].iloc[i-1] >= 30000 and
            70 <= df['close'].iloc[i-1] <= 5000 and
            df['close'].iloc[i-1] <= df['open'].iloc[i-1] and
            df['close'].iloc[i-1] <= df['vwap'].iloc[i-1] and
            df['adx'].iloc[i-1] >= 26
        ):
            signals.append({
                "Stock Name": stock_name,
                "Time": df["time"].iloc[i],
                "Open Price": df["open"].iloc[i]
            })
    return signals

# Main function
def main():
    input_folder = r"D:\DATA_Stocks"
    output_path = r"D:\DATA_Stocks\RESULTS\backtest_results.xlsx"
    all_files = glob.glob(os.path.join(input_folder, "*.xlsx"))
    
    all_signals = []
    for file_path in all_files:
        stock_name = os.path.basename(file_path).replace(".xlsx", "")
        df = load_data(file_path)
        signals = run_backtest(df, stock_name)
        all_signals.extend(signals)
    
    signals_df = pd.DataFrame(all_signals)
    signals_df.to_excel(output_path, index=False)
    print(f"Backtest completed. Results saved in {output_path}")

if __name__ == "__main__":
    main()


In [ ]:
import os
import pandas as pd

# Folder containing Excel files
folder_path = r"D:\DATA_Stocks"

# Iterate through all files in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith(".xlsx") or file_name.endswith(".xls"):  # Process only Excel files
        file_path = os.path.join(folder_path, file_name)
        
        # Read the Excel file
        df = pd.read_excel(file_path)
        
        # Drop last four columns
        df = df.iloc[:, :-4]
        
        # Save the modified file, overwriting the original
        df.to_excel(file_path, index=False)
        print(f"Processed: {file_name}")

print("All files processed successfully.")


In [ ]:
import os
import pandas as pd

# Folder containing Excel files
folder_path = r"D:\DATA_Stocks"

# Iterate through all files in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith(".xlsx") or file_name.endswith(".xls"):  # Process only Excel files
        file_path = os.path.join(folder_path, file_name)
        
        try:
            # Use openpyxl for .xlsx and xlrd for .xls
            engine = "openpyxl" if file_name.endswith(".xlsx") else "xlrd"
            
            # Read the Excel file
            df = pd.read_excel(file_path, engine=engine)
            
            # Reverse the order of rows
            df = df[::-1]
            
            # Save the modified file, overwriting the original
            df.to_excel(file_path, index=False, engine=engine)
            print(f"Processed: {file_name}")
        
        except Exception as e:
            print(f"Error processing {file_name}: {e}")

print("Processing complete.")


In [ ]:
import os
import pandas as pd
import ta

# Folder containing Excel files
folder_path = r"D:\DATA_Stocks"

# Iterate through all files in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith(".xlsx") or file_name.endswith(".xls"):  # Process only Excel files
        file_path = os.path.join(folder_path, file_name)
        
        try:
            # Use openpyxl for .xlsx and xlrd for .xls
            engine = "openpyxl" if file_name.endswith(".xlsx") else "xlrd"
            
            # Read the Excel file
            df = pd.read_excel(file_path, engine=engine)
            
            # Reverse the order of rows
            df = df[::-1]
            
            # Calculate ADX, RSI, and Bollinger Bands
            df['ADX'] = ta.trend.adx(df['high'], df['low'], df['close'])
            df['RSI'] = ta.momentum.rsi(df['close'])
            df['BB_Upper'] = ta.volatility.bollinger_hband(df['close'])
            
            # Save the modified file, overwriting the original
            df.to_excel(file_path, index=False, engine=engine)
            print(f"Processed: {file_name}")
        
        except Exception as e:
            print(f"Error processing {file_name}: {e}")

print("Processing complete.")


In [ ]:
import os
import pandas as pd
import ta

# Folder containing Excel files
folder_path = r"D:\DATA_Stocks"
result_path = r"D:\DATA_Stocks\RESULTS"
result_file = os.path.join(result_path, "final_results.xlsx")

# Ensure result directory exists
os.makedirs(result_path, exist_ok=True)

# List to store results from all files
all_results = []

# Iterate through all files in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith(".xlsx") or file_name.endswith(".xls"):  # Process only Excel files
        file_path = os.path.join(folder_path, file_name)
        
        try:
            # Use openpyxl for .xlsx and xlrd for .xls
            engine = "openpyxl" if file_name.endswith(".xlsx") else "xlrd"
            
            # Read the Excel file
            df = pd.read_excel(file_path, engine=engine)
            
            # Reverse the order of rows
            df = df[::-1]
            
            # Calculate ADX, RSI, and Bollinger Bands
            df['ADX'] = ta.trend.adx(df['high'], df['low'], df['close'])
            df['RSI'] = ta.momentum.rsi(df['close'])
            df['BB_Upper'] = ta.volatility.bollinger_hband(df['close'])
            
            # Filter for specific timestamps
            df['time'] = pd.to_datetime(df['time'])
            df = df[df['time'].dt.strftime('%H:%M').isin(['09:45', '10:10', '11:00'])]
            
            # Check conditions
            df['Condition_Met'] = (
                (df['low'].shift(2) >= df['BB_Upper'].shift(2)) &
                (df['high'].shift(2) >= df['BB_Upper'].shift(2)) &
                (df['RSI'].shift(1) >= 60) &
                (df['open'].shift(6) / df['close'].shift(1) >= 0.955) &
                (df['close'].shift(1) <= df['BB_Upper'].shift(1)) &
                (df['volume'].shift(1) >= 30000) &
                (df['close'].shift(1) >= 70) &
                (df['close'].shift(1) <= 5000) &
                (df['close'].shift(1) <= df['open'].shift(1)) &
                (df['close'].shift(1) <= df['vwap'].shift(1)) &
                (df['ADX'].shift(1) >= 26)
            )
            
            # Keep only rows where all conditions are met
            df = df[df['Condition_Met']]
            
            if not df.empty:
                # Add filename as a column
                df['File'] = file_name
                
                # Store results
                all_results.append(df)
                print(f"Accepted: {file_name}")
            else:
                print(f"Rejected: {file_name} (No matching rows)")
        
        except Exception as e:
            print(f"Error processing {file_name}: {e}")

# Combine all results and save to a single Excel file
if all_results:
    final_df = pd.concat(all_results, ignore_index=True)
    final_df.to_excel(result_file, index=False, engine="openpyxl")
    print(f"Final results saved to {result_file}")
else:
    print("No results to save.")

print("Processing complete.")